# Dataset Exploration: WISDM (Wireless Sensor Data Mining)

**Semester Project Deliverable 1** | ICS 3202, Aug-Nov 2026

## Research area
Fall detection and activity monitoring for elderly care, using smartphone accelerometer data. The end product is a mobile app that reads a phone's motion sensors, classifies what a person is doing (walking, sitting, lying down, and so on), and can flag a likely fall to a caregiver or family member.

## Why this dataset
The WISDM dataset (Kwapisz et al., Fordham University) contains raw triaxial accelerometer readings collected from Android phones carried by 36 subjects performing six activities: Walking, Jogging, Upstairs, Downstairs, Sitting, and Standing. It matches the project's technical solution directly, since the final app will read the same kind of sensor stream from a phone in real time.

**Expected output variable:** `activity` - the class label the ML engine will learn to predict from a window of accelerometer readings. In the finished app, this becomes the signal used to detect risky activity or a fall.

**Source:** Kwapisz, J. R., Weiss, G. M., & Moore, S. A. (2011). *Activity Recognition using Cell Phone Accelerometers*. WISDM Lab, Fordham University. Retrieved from a public mirror on Hugging Face: https://huggingface.co/datasets/stark0908/WISDM (raw file: `WISDM_ar_v1.1_raw.txt`).

In [1]:
import os
import pandas as pd
import numpy as np
import urllib.request

# Download the raw WISDM accelerometer file (public dataset, no auth needed)
url = "https://huggingface.co/datasets/stark0908/WISDM/resolve/main/WISDM_ar_v1.1_raw.txt"
local_path = "WISDM_ar_v1.1_raw.txt"
if not os.path.exists(local_path):
    urllib.request.urlretrieve(url, local_path)

columns = ["user", "activity", "timestamp", "x-accel", "y-accel", "z-accel"]

# The raw file ends each line with a trailing ';' and a handful of rows are malformed
# (missing z-accel, stray commas). We read it as text first so bad rows can be skipped
# instead of silently corrupting the dataframe.
rows = []
with open(local_path, "r") as f:
    for line in f:
        line = line.strip().rstrip(";")
        if not line:
            continue
        parts = line.split(",")
        if len(parts) == 6:
            rows.append(parts)

df = pd.DataFrame(rows, columns=columns)

# Cast numeric columns; anything that fails to parse becomes NaN so it shows up
# honestly in the completeness check rather than being hidden.
df["user"] = pd.to_numeric(df["user"], errors="coerce")
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
for col in ["x-accel", "y-accel", "z-accel"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.head()

,user,activity,timestamp,x-accel,y-accel,z-accel
0,33,Jogging,49105962326000,-0.694638,12.680544,0.503953
1,33,Jogging,49106062271000,5.012288,11.264028,0.953424
2,33,Jogging,49106112167000,4.903325,10.882658,-0.081722
3,33,Jogging,49106222305000,-0.612916,18.496431,3.023717
4,33,Jogging,49106332290000,-1.184970,12.108489,7.205164


## a. How many rows and columns are contained in the dataset? (3 marks)

In [2]:
n_rows, n_cols = df.shape
print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")
print(f"Column names: {list(df.columns)}")

Rows: 1086466
Columns: 6
Column names: ['user', 'activity', 'timestamp', 'x-accel', 'y-accel', 'z-accel']


## b. What datatypes are contained in the dataset? (3 marks)

In [3]:
df.dtypes

user           int64
activity         str
timestamp      int64
x-accel      float64
y-accel      float64
z-accel      float64
dtype: object

## c. Is the dataset complete? i.e. no missing values? (2 marks)

In [4]:
missing_counts = df.isnull().sum()
print(missing_counts)
print()
total_missing = missing_counts.sum()
if total_missing == 0:
    print("The dataset is complete: no missing values in any column.")
else:
    print(f"The dataset is NOT complete: {total_missing} missing values found, "
          f"mostly from a small number of malformed rows in the raw sensor log "
          f"(a known quirk of this WISDM release, where a z-accel reading is "
          f"occasionally dropped at the source).")

user         0
activity     0
timestamp    0
x-accel      0
y-accel      0
z-accel      1
dtype: int64

The dataset is NOT complete: 1 missing values found, mostly from a small number of malformed rows in the raw sensor log (a known quirk of this WISDM release, where a z-accel reading is occasionally dropped at the source).


## d. Slice out the first 15 rows and last 20 rows, merge into `df_sample`, and display it (4 marks)

In [5]:
first_15 = df.head(15)
last_20 = df.tail(20)

df_sample = pd.concat([first_15, last_20])
df_sample

,user,activity,timestamp,x-accel,y-accel,z-accel
0,33,Jogging,49105962326000,-0.694638,12.680544,0.503953
1,33,Jogging,49106062271000,5.012288,11.264028,0.953424
2,33,Jogging,49106112167000,4.903325,10.882658,-0.081722
3,33,Jogging,49106222305000,-0.612916,18.496431,3.023717
4,33,Jogging,49106332290000,-1.184970,12.108489,7.205164
5,33,Jogging,49106442306000,1.375655,-2.492524,-6.510526
6,33,Jogging,49106542312000,-0.612916,10.569390,5.706926
7,33,Jogging,49106652389000,-0.503953,13.947236,7.055340
8,33,Jogging,49106762313000,-8.430995,11.413852,5.134871
9,33,Jogging,49106872299000,0.953424,1.375655,1.648062
